# Credit vs Manual reconciliation (live trades)

`Credit` = official booking (flows downstream from official booking systems); `Manual` = trades we book ourselves.
Trade COUNTS need not match (2 x 50mm == 1 x 100mm) but net **notional, PV and Spread01 per risk bucket must tie**.

Status per bucket: `OK` ties, `BREAK` both strategies have it but notional/Spread01 differ, `ONLY_Credit` / `ONLY_Manual` bucket missing from one side, `INCOMPLETE` a trade did not load/price.

Needs the pricer cache for `AS_OF` (build it first, see `liverisk_testing.ipynb` or `python pricer_cache.py build --as-of <date> --live-only`). **Restart the kernel after any qncore install.**

In [ ]:
import os, time
import pandas as pd
import requests
from Quantifi.Toolkit.Base import Dt
from qservices.QuantifiApiClient import *
import pricer_cache as pc
import credit_manual_rec as rec

ENV = 'Prod'
host = {'Prod': 'hb-dc1pd-qfidsp', 'UAT': 'hb-dc1ut-qfidsp'}[ENV]
os.environ['Q_DATA_SERVICE_URL_PREFIX'] = f'https://{host}:47200'
os.environ['Q_MARKET_ENV_SERVICE_URL_PREFIX'] = f'https://{host}:47201'
os.environ['Q_LOCAL_INSTALL'] = 'True'
requests.packages.urllib3.disable_warnings()

AS_OF = '2026-11-13'
CALC_ENV = 'Official'
TRADERS = ['abhutra', 'AB', 'Abhishek Bhutra']
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 300)
pd.options.display.float_format = '{:,.2f}'.format

## 1. Live Credit / Manual trades

In [ ]:
tl = ', '.join(f"'{t}'" for t in TRADERS)
all_trades = Connect.data_query(f"FROM Trade b WHERE b.Trader.Name IN ({tl})", max_results=100_000, verify_https_cert=False)
live = rec.live_static(all_trades, AS_OF)
print(len(all_trades), 'trades;', len(live), 'live in Credit/Manual')
live.groupby(['ptype', 'strategy']).size().unstack(fill_value=0)

## 2. Pricers (from cache) and per-trade risk
PV and Spread01 (+1bp, all survival curves, `Sensitivities.Spread01`). A live trade that is not in the cache shows up as `not loaded`.

In [ ]:
pricers = rec.load_pricers(live, AS_OF, CALC_ENV)
risk = rec.trade_risk(live, pricers)
print(len(pricers), 'pricers;', risk.error.notna().sum(), 'trades without risk')
risk[risk.error.notna()][['TradeId', 'strategy', 'ptype', 'bucket', 'error']]

## 3. Reconciliation by risk bucket

In [ ]:
rep = rec.reconcile(risk)
rep.groupby(['ptype', 'status']).size().unstack(fill_value=0)

In [ ]:
cols = ['ptype', 'bucket', 'trades_Credit', 'trades_Manual', 'notional_Credit', 'notional_Manual', 'notional_diff',
        'pv_Credit', 'pv_Manual', 'pv_diff', 'spread01_Credit', 'spread01_Manual', 'spread01_diff', 'status']
rep[rep.status != 'OK'][cols]

### Totals by strategy and product (PV and Spread01)

In [ ]:
tot = risk.groupby(['ptype', 'strategy'])[['notional', 'pv', 'spread01']].sum().unstack('strategy')
tot

### CDX options (priority product)
Short-dated, so these are the ones where Credit/Manual must tie most tightly.

In [ ]:
rep[rep.ptype == 'CDXOption'][cols]

In [ ]:
out = f'credit_manual_rec_{AS_OF}.xlsx'
with pd.ExcelWriter(out) as xw:
    rep.to_excel(xw, 'buckets', index=False)
    risk.to_excel(xw, 'trades', index=False)
print('saved', out)